<a href="https://colab.research.google.com/github/zelal-Eizaldeen/deeplearning_course/blob/main/streaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# LLM Streaming Serving Basics

In our vLLM code example, you might have noticed that although the LLM generates tokens one by one internally, the call to outputs = llm.generate([prompt], inference_params) waits until the entire output is generated before returning any result. In the context of a web service—for example, a chatbot—this means the user experiences a long delay before receiving any response–from a few seconds to even minutes, depending on the output length.

Tokens are generated sequentially during the **Decoding phase**. To improve responsiveness, you can return each token immediately as it is generated, which is a technique known as LLM **streaming**.

 **Streaming**, in LLMs, refers to the **process of incrementally returning output token-by-token (or in small chunks) during generation, rather than waiting for the entire output to complete.**

In [ ]:
!pip install --quiet vllm transformers tiktoken


Streaming:

- Return result when generation completes.
- Return as soon as we have a token.

The key difference in the streaming version of the vLLM code is that we initialize the model using the **AsyncLLMEngine** class instead of the standard LLM class. With **AsyncLLMEngine**, the generate() function returns an **asynchronous stream object (AsyncStream) that allows us to pull newly generated tokens one by one using an async for** loop

—for example, async for request_output in results_generator.

In [ ]:
import asyncio
from vllm.engine.arg_utils import AsyncEngineArgs
from vllm.engine.async_llm_engine import AsyncLLMEngine
from vllm.sampling_params import SamplingParams

# Initialize the engine arguments
engine_args = AsyncEngineArgs(
    model="Qwen/Qwen2.5-0.5B",
    dtype="float16",
    tensor_parallel_size=1,      # Number of GPUs to use
    gpu_memory_utilization=0.9,  # GPU memory utilization
    max_num_batched_tokens=32768, # Maximum number of tokens to process in a batch
    max_num_seqs=256,           # Maximum number of sequences to process
    disable_log_stats=True,      # Disable stats logging
)

# Create the vLLM async streaming engine
engine = AsyncLLMEngine.from_engine_args(engine_args)

In [5]:
async def generate_text(prompt: str, max_tokens: int = 100, request_id="id"):
  try:
    # Define sampling parameters
    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=max_tokens,
        stop=["\n"],  # Stop at newline
    )

    # Generate text in async and streaming fashion
    results_generator = engine.generate(
        prompt=prompt,
        sampling_params=sampling_params,
        request_id=request_id
    )

    # Process the results
    final_output = None
    async for request_output in results_generator:
        final_output = request_output
        # Print each token as it's generated
        print("chunk \n")
        for output in request_output.outputs:
            print(output.text, end="", flush=True)
        print()
    print()  # Newline at the end

    # This will only be reached if all tokens are generated
    print("\nGeneration completed successfully")

    return final_output
  except asyncio.CancelledError:
    print("\nGeneration was cancelled")
    return None
  finally:
    # Always clean up
    try:
      await engine.abort(request_id)
    except:
      pass

In [ ]:
import uuid
async def main():
    """Defines the main asynchronous execution flow."""
    # 1. Define the input prompt and parameters
    prompt = "The capital of Syria is"

    # 2. Generate a unique request ID (best practice)
    request_id = str(uuid.uuid4())

    # 3. Set a reasonable max_tokens limit
    max_tokens_limit = 120

    print(f"Starting generation for request ID: {request_id}")
    print(f"Prompt: {prompt}\n")

    # 4. Call the async generation function
    result = await generate_text(prompt, max_tokens_limit, request_id)

    if result:
        print(f"\n--- Final Output Summary ---")
        print(f"Stop Reason: {result.outputs[0].finish_reason}")
        print(f"Generated Tokens: {len(result.outputs[0].token_ids)}")


# --- EXECUTION ---
await main()




The word "chunk" in your output refers to a partial segment of text (one or more tokens) that the vLLM engine processed and sent back to your Python script during the text generation process.

This is the streaming behaviour and is a direct result of how you designed your generate_text function.

Another valuable benefit of streaming is that it allows users to cancel generation midway if the output is heading in an undesired direction. In vLLM, this can be achieved by calling engine.abort(request_id), where request_id is a unique identifier associated with the specific generation request. This feature not only improves the user experience by avoiding irrelevant or incorrect completions, it also helps conserve compute resources, which is especially important in production environments where efficiency and cost control are critical.